In [1]:
import os
import re
import pickle
import pandas as pd
from dotenv import load_dotenv
from groq import Groq

os.chdir(r"C:\Users\Landl\Downloads\Data-Science-Projects\sec-financial-nlp")
load_dotenv(override=True)

with open("data/processed/all_texts.pkl", "rb") as f:
    all_texts_fixed = pickle.load(f)

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

print(f"Filings loaded: {len(all_texts_fixed)}")
print(f"Groq client ready: {client is not None}")

Filings loaded: 125
Groq client ready: True


In [3]:
def chunk_filing(text, chunk_size=350, overlap=75):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

In [2]:
import chromadb
from sentence_transformers import SentenceTransformer

chroma_client = chromadb.Client()
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print('done')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

done


In [5]:
def build_ticker_collection(ticker, all_texts_fixed, chroma_client, embed_model):
    collection_name = f"filings_{ticker}"
    try:
        chroma_client.delete_collection(collection_name)
    except Exception:
        pass
    collection = chroma_client.create_collection(collection_name)
    ticker_keys = [k for k in all_texts_fixed if k.startswith(f"{ticker}_10-K")]
    all_chunks, all_ids, all_metadata = [], [], []
    for key in ticker_keys:
        text = all_texts_fixed[key]
        chunks = chunk_filing(text)
        for i, chunk in enumerate(chunks):
            all_chunks.append(chunk)
            all_ids.append(f"{key}_chunk{i}")
            all_metadata.append({"filing": key, "chunk_index": i})
    embeddings = embed_model.encode(all_chunks, show_progress_bar=True)
    collection.add(documents=all_chunks, embeddings=embeddings.tolist(), ids=all_ids, metadatas=all_metadata)
    print(f"{ticker}: {len(ticker_keys)} filings, {len(all_chunks)} chunks indexed")
    return collection
print("done")

done


In [6]:
# test
aapl_collection = build_ticker_collection("AAPL", all_texts_fixed, chroma_client, embed_model)

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

AAPL: 5 filings, 420 chunks indexed


In [7]:
def retrieve_chunks(question, collection, embed_model, top_k=7):
    """
    Embeds the user's question and retrieves the top_k most similar
    chunks from the given ticker's ChromaDB collection.
    """
    question_embedding = embed_model.encode([question]).tolist()
    results = collection.query(
        query_embeddings=question_embedding,
        n_results=top_k
    )
    return results["documents"][0], results["metadatas"][0]

In [8]:
def generate_answer(question, docs, client, ticker):
    context = "\n\n---\n\n".join(docs)
    prompt = f"""You are a financial analyst assistant. Answer the question using ONLY the context below, from {ticker}'s SEC 10-K filings across multiple years.

Keep your answer to 2-3 sentences maximum. Be direct and specific — no preamble, no filler.
If the context contains data from multiple years, clearly state which year each figure is from — do not blend figures from different years into one answer.
If the answer is not present in the context, say "This information was not found in the provided filing excerpts" and say nothing else.

Context:
{context}

Question: {question}

Answer:"""
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=180,
        temperature=0.1
    )
    return response.choices[0].message.content.strip()

In [9]:
collections = {}
for ticker in ["AAPL", "MSFT", "GOOGL", "JPM", "TSLA"]:

    collections[ticker] = build_ticker_collection(ticker, all_texts_fixed, chroma_client, embed_model)

Batches:   0%|          | 0/14 [00:00<?, ?it/s]

AAPL: 5 filings, 420 chunks indexed


Batches:   0%|          | 0/21 [00:00<?, ?it/s]

MSFT: 5 filings, 665 chunks indexed


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

GOOGL: 5 filings, 612 chunks indexed


Batches:   0%|          | 0/60 [00:00<?, ?it/s]

JPM: 5 filings, 1914 chunks indexed


Batches:   0%|          | 0/25 [00:00<?, ?it/s]

TSLA: 5 filings, 788 chunks indexed


In [10]:
test_questions = {
    "AAPL": [
        "What is Apple's main business risk related to China?",
        "What percentage of Apple's revenue comes from Services?",  # specific number - tests whether it finds it or correctly says not found
    ],
    "MSFT": [
        "What are Microsoft's key risks related to cloud computing and Azure?",
        "What was Microsoft's exact net income in fiscal year 2021?",  # specific figure - hard test
    ],
    "GOOGL": [
        "What antitrust or regulatory risks does Google face?",
        "How does Google describe its approach to artificial intelligence investment?",  # vague/broad, multiple sections likely relevant
    ],
    "JPM": [
        "What does JPMorgan say about regulatory capital requirements?",
        "What is JPMorgan's dividend policy?",  # may or may not be covered in 10-K risk-heavy sections
    ],
    "TSLA": [
        "What risks does Tesla describe related to battery supply?",
        "What did Tesla say about its stock price volatility risk?",
    ],
}

for ticker, questions in test_questions.items():
    print(f"\n{'='*60}\n{ticker}\n{'='*60}")
    for q in questions:
        docs, metas = retrieve_chunks(q, collections[ticker], embed_model, top_k=5)
        answer = generate_answer(q, docs, client, ticker)
        print(f"\nQ: {q}")
        print(f"A: {answer}")


AAPL

Q: What is Apple's main business risk related to China?
A: Apple's main business risk related to China is the impact of trade policies and disputes, including tariffs and other measures that restrict international trade, which can materially adversely affect the Company's business, particularly since substantially all of its manufacturing is performed in whole or in part by outsourcing partners located primarily in China mainland, as well as other countries in Asia. This risk is noted in the 2022, 2023, and 2024 Form 10-K filings.

Q: What percentage of Apple's revenue comes from Services?
A: This information was not found in the provided filing excerpts.

MSFT

Q: What are Microsoft's key risks related to cloud computing and Azure?
A: Microsoft's key risks related to cloud computing and Azure include excessive outages, data losses, and disruptions of online services, as well as operational failures, including temporary or permanent loss of customer data, insufficient Internet c

In [13]:
rag_test_results = []

for ticker, questions in test_questions.items():
    for q in questions:
        docs, metas = retrieve_chunks(q, collections[ticker], embed_model, top_k=7)
        answer = generate_answer(q, docs, client, ticker)
        source_filings = ", ".join(sorted(set(m["filing"] for m in metas)))
        rag_test_results.append({
            "ticker": ticker,
            "question": q,
            "answer": answer,
            "source_filings": source_filings,
            "chunk_size": 350,
            "top_k": 7
        })

df_rag_tests = pd.DataFrame(rag_test_results)
df_rag_tests.to_csv("data/processed/rag_test_results.csv", index=False)
print(f"Saved {len(df_rag_tests)} test Q&A pairs to data/processed/rag_test_results.csv")
df_rag_tests

Saved 10 test Q&A pairs to data/processed/rag_test_results.csv


,ticker,question,answer,source_filings,chunk_size,top_k
0,AAPL,What is Apple's main business risk related to ...,This information was not found in the provided...,"AAPL_10-K_0000320193-20-000096, AAPL_10-K_0000...",350,7
1,AAPL,What percentage of Apple's revenue comes from ...,This information was not found in the provided...,"AAPL_10-K_0000320193-20-000096, AAPL_10-K_0000...",350,7
2,MSFT,What are Microsoft's key risks related to clou...,Microsoft's key risks related to cloud computi...,"MSFT_10-K_0000950170-23-035122, MSFT_10-K_0001...",350,7
3,MSFT,What was Microsoft's exact net income in fisca...,This information was not found in the provided...,"MSFT_10-K_0000950170-23-035122, MSFT_10-K_0000...",350,7
4,GOOGL,What antitrust or regulatory risks does Google...,"Google faces antitrust complaints, such as the...","GOOGL_10-K_0001652044-20-000008, GOOGL_10-K_00...",350,7
5,GOOGL,How does Google describe its approach to artif...,Google describes its approach to artificial in...,"GOOGL_10-K_0001652044-20-000008, GOOGL_10-K_00...",350,7
6,JPM,What does JPMorgan say about regulatory capita...,JPMorgan Chase is subject to various regulator...,"JPM_10-K_0000019617-20-000257, JPM_10-K_000001...",350,7
7,JPM,What is JPMorgan's dividend policy?,Federal law imposes limitations on the payment...,"JPM_10-K_0000019617-20-000257, JPM_10-K_000001...",350,7
8,TSLA,What risks does Tesla describe related to batt...,Tesla describes risks related to battery suppl...,"TSLA_10-K_0000950170-22-000796, TSLA_10-K_0000...",350,7
9,TSLA,What did Tesla say about its stock price volat...,Tesla stated that its stock price may be volat...,"TSLA_10-K_0000950170-22-000796, TSLA_10-K_0001...",350,7
